In [ ]:
import os
from PIL import Image
import numpy as np
from keras import backend as K
from keras.preprocessing.image import img_to_array
from sklearn.model_selection import train_test_split
import cv2
from keras.preprocessing.image import ImageDataGenerator

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


In [ ]:
# img_data = os.listdir('./data/small_data/train')
img_data = os.listdir('./data/small_data/train')

flower_mapping = {'daisy':0, 'dandelion':1, 'rose':2, 'sunflower':3, 'tulip':4}

IMAGE_DIMS = (240, 320, 3)
data = []
labels = []


# loop over the input images
# for imagePath in imagePaths:
# 	# load the image, pre-process it, and store it in the data list
# 	image = cv2.imread(imagePath)
# 	image = cv2.resize(image, (IMAGE_DIMS[1], IMAGE_DIMS[0]))
# 	image = img_to_array(image)
# 	images.append(image)
 
# 	# extract the class label from the image path and update the
# 	# labels list
# 	label = imagePath.split(os.path.sep)[-2]
# 	labels.append(label)

for img_dir in img_data:
    for fn in os.listdir(os.path.join('./data/small_data/train/',img_dir)):
        fd = os.path.join('./data/small_data/train/',img_dir,fn)
        image = cv2.imread(fd)
        image = cv2.resize(image, (IMAGE_DIMS[1], IMAGE_DIMS[0]))
        image = img_to_array(image)
        data.append(image)
        labels.append(flower_mapping.get(img_dir))
        
# for img_dir in img_data:
#     for fn in os.listdir(os.path.join('./data/image_data/train/',img_dir)):
#         fd = os.path.join('./data/image_data/train/',img_dir,fn)
#         image = cv2.imread(fd)
#         image = cv2.resize(image, (IMAGE_DIMS[1], IMAGE_DIMS[0]))
#         image = img_to_array(image)
#         data.append(image)
#         labels.append(flower_mapping.get(img_dir))

In [ ]:
data[0].shape

In [ ]:
data_generator = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True)

In [ ]:
data = np.array(data, dtype="float") / 255.0
labels = np.array(labels)
print("[INFO] data matrix: {:.2f}MB".format(
	data.nbytes / (1024 * 1000.0)))

In [ ]:
(trainX, testX, trainY, testY) = train_test_split(data,
	labels, test_size=0.2, random_state=42)

In [ ]:
trainX = next(data_generator.flow(trainX, shuffle=False))
trainY = next(data_generator.flow(trainY, shuffle=False))


In [ ]:
EPOCHS = 20
INIT_LR = 1e-3
BS = 32

In [ ]:
testX.shape

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

aug = ImageDataGenerator(rotation_range=25, width_shift_range=0.1,
	height_shift_range=0.1, shear_range=0.2, zoom_range=0.2,
	horizontal_flip=True, fill_mode="nearest")

In [ ]:
##V0 CNN

# from keras.models import Sequential
# from keras.layers import Dense, Dropout, Activation, Flatten
# from keras.layers import Conv2D, MaxPooling2D
# from keras.optimizers import RMSprop, Adam

# num_classes = 5
# testY = keras.utils.to_categorical(testY, num_classes)
# trainY = keras.utils.to_categorical(trainY, num_classes)

# model = Sequential()
# model.add(Conv2D(16, (5, 5), padding='same',
#                  input_shape=trainX.shape[1:]))
# model.add(Activation('relu'))
# model.add(Conv2D(16, (5, 5)))
# model.add(Activation('relu'))
# model.add(MaxPooling2D(pool_size=(2, 2)))

# model.add(Conv2D(32, (5, 5), padding='same'))
# model.add(Activation('relu'))
# model.add(Conv2D(32, (5, 5)))
# model.add(Activation('relu'))
# model.add(MaxPooling2D(pool_size=(2, 2)))

# model.add(Flatten())
# model.add(Dropout(0.5))
# model.add(Dense(num_classes))
# model.add(Activation('softmax'))
# model.summary()

##SmallVGGNet
from keras.models import Sequential
from keras.layers.normalization import BatchNormalization
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers.core import Activation
from keras.layers.core import Flatten
from keras.layers.core import Dropout
from keras.layers.core import Dense

classes = 5
model = Sequential()
inputShape = trainX.shape[1:]
chanDim = -1
testY = keras.utils.to_categorical(testY, num_classes)
trainY = keras.utils.to_categorical(trainY, num_classes)
inputShape = trainX.shape[1:]

model.add(Conv2D(32, (3,3), padding="same", input_shape=inputShape))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(3,3)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3,3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(Conv2D(64, (3,3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3,3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(Conv2D(128, (3,3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(1024))
model.add(Activation("relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(classes))
model.add(Activation("sigmoid"))
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer=Adam(),
              metrics=['accuracy'])
history = model.fit(trainX, trainY,
                    batch_size=4,
                    epochs=EPOCHS,
                    verbose=1,
                    validation_data=(testX, testY))
score = model.evaluate(testX, testY, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
test_img_data = os.listdir('./data/image_data/test')
test_data = []
for test_img_dir in test_img_data:
    fd = os.path.join('./data/image_data/test/',test_img_dir)
    image = cv2.imread(fd)
    image = cv2.resize(image, (IMAGE_DIMS[1], IMAGE_DIMS[0]))
    image = img_to_array(image)
    test_data.append(image)
test_data[0].shape

In [ ]:
test_data = np.array(test_data, dtype="float") / 255.0
print("[INFO] data matrix: {:.2f}MB".format(
	test_data.nbytes / (1024 * 1000.0)))

In [ ]:
ynew = model.predict(test_data)

In [ ]:
ynew

In [ ]:
res = []
for yn in ynew:
    res.append(np.argmax(yn))
for r in res:
    print(r)

In [ ]:
import pandas as pd 
test_img_data = os.listdir('./data/image_data/test')
test_id = []
i=0
for test_img_dir in test_img_data:
    test_id.append([test_img_dir.split(".")[0],res[i]])
    i = i +1

np_ids= np.array(test_id)
test_ids = pd.DataFrame(np_ids)
test_ids.columns = ['id', 'flower_class']
test_ids

In [ ]:
dir_data = './'
test_ids.to_csv(dir_data+'final_ans.csv', encoding='utf-8', index=False)